# Experiment Summary

This notebook implements and evaluates classical machine learning approaches for sarcasm and irony detection using TF-IDF feature extraction and Support Vector Machine (SVM) classifiers.

The experiments were conducted to address the following research questions:

---

## RQ1 — Baseline Sarcasm and Irony Detection Performance

The first set of experiments evaluated the effectiveness of TF-IDF + SVM models on individual datasets using in-domain training and testing.

### Datasets Used
- Reddit sarcasm dataset
- News Headlines sarcasm dataset
- SemEval irony dataset

### Workflow
For each dataset:
1. Load preprocessed train/test splits
2. Load the corresponding TF-IDF vectorizer
3. Transform text into TF-IDF feature vectors
4. Train a Linear SVM classifier
5. Evaluate performance using:
   - Accuracy
   - Precision
   - Recall
   - F1-score
   - Confusion Matrix
6. Save trained models as `.pkl` files

### In-Domain Experiments
- Reddit → Reddit
- Headlines → Headlines
- SemEval → SemEval

---

## RQ2 — Effect of Contextual Information

The second set of experiments investigated whether incorporating contextual information improves sarcasm detection performance.

### Experiment A — Text Only
The model was trained using only the target text:
- `text`

### Experiment B — Text + Context
The model was trained using:
- `text_with_context`

A separate TF-IDF vectorizer was created for the context-based representation.

### Findings
The TF-IDF + SVM model using contextual information did not improve performance on the Reddit dataset. The text-only representation achieved better overall metrics, suggesting that TF-IDF may not effectively capture semantic contextual relationships.

---

## RQ3 — Cross-Domain Generalisation

The final set of experiments evaluated how well sarcasm and irony detection models generalize across domains.

### Cross-Domain Setup
Models were trained on one dataset and evaluated on another dataset without retraining.

### Experiments Conducted
- Reddit → Headlines
- Headlines → Reddit
- Reddit → SemEval
- SemEval → Reddit
- Headlines → SemEval
- SemEval → Headlines

### Key Observations
- Cross-domain performance was significantly lower than in-domain performance.
- Transferability depended on:
  - dataset size,
  - linguistic similarity,
  - writing style,
  - and vocabulary overlap.
- Models trained on Reddit generalized better overall due to the larger and more diverse training data.
- TF-IDF + SVM models struggled to learn generalized semantic representations across domains.

---

## Saved Models

The following artifacts were generated and saved in the `/models` directory:
- TF-IDF vectorizers (`tfidf_*.pkl`)
- Trained SVM models (`svm_*.pkl`)

These saved models can later be reused for:
- prediction,
- evaluation,
- deployment,
- and comparison with deep learning approaches.

In [3]:
import pandas as pd
import joblib

from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [25]:
# Train
def train_svm_model(
    train_path,
    test_path,
    tfidf_path,
    model_save_path,
    text_column="text"
):
    
    import pandas as pd
    import joblib

    from sklearn.svm import LinearSVC

    from sklearn.metrics import (
        accuracy_score,
        precision_score,
        recall_score,
        f1_score,
        classification_report,
        confusion_matrix
    )

    
    # Load datasets
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    # Features and labels
    X_train_text = train_df[text_column]
    y_train = train_df["label"]

    X_test_text = test_df[text_column]
    y_test = test_df["label"]

    # Load TF-IDF vectorizer
    tfidf = joblib.load(tfidf_path)

    # Transform text
    X_train = tfidf.transform(X_train_text)
    X_test = tfidf.transform(X_test_text)

    
    # Train SVM
    svm_model = LinearSVC(
        class_weight="balanced",
        random_state=42,
        max_iter=5000
    )

    svm_model.fit(X_train, y_train)

    # Predictions
    y_pred = svm_model.predict(X_test)

    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print("=" * 50)
    print("MODEL PERFORMANCE")
    print("=" * 50)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-Score : {f1:.4f}")

    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))

    print("\nConfusion Matrix:\n")
    print(confusion_matrix(y_test, y_pred))


    # Save Model
    joblib.dump(svm_model, model_save_path)

    print("\nModel saved successfully!")

## Experiment A: Using TEXT only

In [26]:
# RQ1: TF-IDF + SVM Performance
# RQ2: Effect of Contextual Information
# Reddit Dataset
print("Reddit (TEXT only)")
train_svm_model(
    "../data/processed/reddit_train.csv",
    "../data/processed/reddit_test.csv",
    "../models/tfidf_reddit.pkl",
    "../models/svm_reddit.pkl",
    text_column="text"
)

MODEL PERFORMANCE
Accuracy : 0.6966
Precision: 0.7193
Recall   : 0.6447
F1-Score : 0.6800

Classification Report:

              precision    recall  f1-score   support

           0       0.68      0.75      0.71    101080
           1       0.72      0.64      0.68    101073

    accuracy                           0.70    202153
   macro avg       0.70      0.70      0.70    202153
weighted avg       0.70      0.70      0.70    202153


Confusion Matrix:

[[75653 25427]
 [35907 65166]]

Model saved successfully!


In [22]:
# RQ1: TF-IDF + SVM Performance
# News Headlines Dataset
print("News Headlines")
train_svm_model(
    "../data/processed/headlines_train.csv",
    "../data/processed/headlines_test.csv",
    "../models/tfidf_headlines.pkl",
    "../models/svm_headlines.pkl"
)

MODEL PERFORMANCE
Accuracy : 0.8356
Precision: 0.8224
Recall   : 0.8354
F1-Score : 0.8288

Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.84      0.84      2997
           1       0.82      0.84      0.83      2727

    accuracy                           0.84      5724
   macro avg       0.84      0.84      0.84      5724
weighted avg       0.84      0.84      0.84      5724


Confusion Matrix:

[[2505  492]
 [ 449 2278]]

Model saved successfully!


In [23]:
# RQ1: TF-IDF + SVM Performance
# SemEval Dataset
print("SemEval")
train_svm_model(
    "../data/processed/semeval_train.csv",
    "../data/processed/semeval_test.csv",
    "../models/tfidf_semeval.pkl",
    "../models/svm_semeval.pkl"
)

MODEL PERFORMANCE
Accuracy : 0.7749
Precision: 0.7518
Recall   : 0.8189
F1-Score : 0.7839

Classification Report:

              precision    recall  f1-score   support

           0       0.80      0.73      0.77       383
           1       0.75      0.82      0.78       381

    accuracy                           0.77       764
   macro avg       0.78      0.77      0.77       764
weighted avg       0.78      0.77      0.77       764


Confusion Matrix:

[[280 103]
 [ 69 312]]

Model saved successfully!


## Experiment B: Using TEXT with CONTEXT

In [27]:
# RQ2: Text vs Text plus Context
# Reddit Dataset
print("Reddit (TEXT + CONTEXT)")
train_svm_model(
    "../data/processed/reddit_train.csv",
    "../data/processed/reddit_test.csv",
    "../models/tfidf_reddit_context.pkl",
    "../models/svm_reddit_context.pkl",
    text_column="text_with_context"
)

MODEL PERFORMANCE
Accuracy : 0.6785
Precision: 0.6989
Recall   : 0.6269
F1-Score : 0.6610

Classification Report:

              precision    recall  f1-score   support

           0       0.66      0.73      0.69    101080
           1       0.70      0.63      0.66    101073

    accuracy                           0.68    202153
   macro avg       0.68      0.68      0.68    202153
weighted avg       0.68      0.68      0.68    202153


Confusion Matrix:

[[73786 27294]
 [37708 63365]]

Model saved successfully!


In [19]:
# Prediction
import joblib

# Load vectorizer
tfidf = joblib.load("../models/tfidf_reddit.pkl")

# Load model
svm_model = joblib.load("../models/svm_reddit.pkl")

# New sample
sample_text = [
    "Yeah because waiting in traffic for 2 hours is super fun"
]

# Transform
sample_vector = tfidf.transform(sample_text)

# Predict
prediction = svm_model.predict(sample_vector)

# Output
if prediction[0] == 1:
    print("Sarcastic")
else:
    print("Not Sarcastic")

Sarcastic


## RQ3: Cross-Domain Generalization

In [30]:
# RQ3: Reddit → Headlines
print("Reddit to Headlines")
train_svm_model(
    "../data/processed/reddit_train.csv",
    "../data/processed/headlines_test.csv",
    "../models/tfidf_reddit.pkl",
    "../models/svm_reddit_to_headlines.pkl",
    text_column="text"
)

Reddit to Headlines
MODEL PERFORMANCE
Accuracy : 0.4677
Precision: 0.4065
Recall   : 0.2552
F1-Score : 0.3136

Classification Report:

              precision    recall  f1-score   support

           0       0.49      0.66      0.57      2997
           1       0.41      0.26      0.31      2727

    accuracy                           0.47      5724
   macro avg       0.45      0.46      0.44      5724
weighted avg       0.45      0.47      0.45      5724


Confusion Matrix:

[[1981 1016]
 [2031  696]]

Model saved successfully!


In [31]:
# RQ3: Headlines → Reddit
print("Headlines to Reddit")
train_svm_model(
    "../data/processed/headlines_train.csv",
    "../data/processed/reddit_test.csv",
    "../models/tfidf_headlines.pkl",
    "../models/svm_headlines_to_reddit.pkl",
    text_column="text"
)

Headlines to Reddit
MODEL PERFORMANCE
Accuracy : 0.4988
Precision: 0.4979
Recall   : 0.2884
F1-Score : 0.3653

Classification Report:

              precision    recall  f1-score   support

           0       0.50      0.71      0.59    101080
           1       0.50      0.29      0.37    101073

    accuracy                           0.50    202153
   macro avg       0.50      0.50      0.48    202153
weighted avg       0.50      0.50      0.48    202153


Confusion Matrix:

[[71683 29397]
 [71922 29151]]

Model saved successfully!


In [32]:
# RQ3: Reddit → SemEval
print("Reddit to SemEval")
train_svm_model(
    "../data/processed/reddit_train.csv",
    "../data/processed/semeval_test.csv",
    "../models/tfidf_reddit.pkl",
    "../models/svm_reddit_to_semeval.pkl",
    text_column="text"
)

Reddit to SemEval
MODEL PERFORMANCE
Accuracy : 0.5877
Precision: 0.6146
Recall   : 0.4646
F1-Score : 0.5291

Classification Report:

              precision    recall  f1-score   support

           0       0.57      0.71      0.63       383
           1       0.61      0.46      0.53       381

    accuracy                           0.59       764
   macro avg       0.59      0.59      0.58       764
weighted avg       0.59      0.59      0.58       764


Confusion Matrix:

[[272 111]
 [204 177]]

Model saved successfully!


In [33]:
# RQ3: SemEval → Reddit
print("SemEval to Reddit")
train_svm_model(
    "../data/processed/semeval_train.csv",
    "../data/processed/reddit_test.csv",
    "../models/tfidf_semeval.pkl",
    "../models/svm_semeval_to_reddit.pkl",
    text_column="text"
)

SemEval to Reddit
MODEL PERFORMANCE
Accuracy : 0.5222
Precision: 0.5830
Recall   : 0.1557
F1-Score : 0.2458

Classification Report:

              precision    recall  f1-score   support

           0       0.51      0.89      0.65    101080
           1       0.58      0.16      0.25    101073

    accuracy                           0.52    202153
   macro avg       0.55      0.52      0.45    202153
weighted avg       0.55      0.52      0.45    202153


Confusion Matrix:

[[89820 11260]
 [85333 15740]]

Model saved successfully!


In [34]:
# RQ3: Headlines → SemEval
print("Headlines to SemEval")
train_svm_model(
    "../data/processed/headlines_train.csv",
    "../data/processed/semeval_test.csv",
    "../models/tfidf_headlines.pkl",
    "../models/svm_headlines_to_semeval.pkl",
    text_column="text"
)

Headlines to SemEval
MODEL PERFORMANCE
Accuracy : 0.5131
Precision: 0.5170
Recall   : 0.3596
F1-Score : 0.4241

Classification Report:

              precision    recall  f1-score   support

           0       0.51      0.67      0.58       383
           1       0.52      0.36      0.42       381

    accuracy                           0.51       764
   macro avg       0.51      0.51      0.50       764
weighted avg       0.51      0.51      0.50       764


Confusion Matrix:

[[255 128]
 [244 137]]

Model saved successfully!


In [35]:
# RQ3: SemEval → Headlines
print("SemEval to Headlines")
train_svm_model(
    "../data/processed/semeval_train.csv",
    "../data/processed/headlines_test.csv",
    "../models/tfidf_semeval.pkl",
    "../models/svm_semeval_to_headlines.pkl",
    text_column="text"
)

SemEval to Headlines
MODEL PERFORMANCE
Accuracy : 0.5152
Precision: 0.4455
Recall   : 0.0719
F1-Score : 0.1238

Classification Report:

              precision    recall  f1-score   support

           0       0.52      0.92      0.66      2997
           1       0.45      0.07      0.12      2727

    accuracy                           0.52      5724
   macro avg       0.48      0.50      0.39      5724
weighted avg       0.49      0.52      0.41      5724


Confusion Matrix:

[[2753  244]
 [2531  196]]

Model saved successfully!
